# YOLOv11 Training — Traffic Control Symbol Detection

Train YOLOv11-s on labeled TCP sheet tiles using a Kaggle GPU notebook.

**Prerequisites:**
- Upload your labeled dataset as a Kaggle dataset (images/ + labels/ in YOLO format)
- Enable GPU accelerator (P100 or T4) in notebook settings
- This notebook clones the repo, installs deps, tiles the data, and trains

## 1. Setup

In [ ]:
# Install uv and clone the repo
!curl -LsSf https://astral.sh/uv/install.sh | sh
!export PATH="$HOME/.local/bin:$PATH" && uv --version

# Clone the repo
!git clone https://github.com/fLu-2/traffic-control-tool.git /kaggle/working/repo
%cd /kaggle/working/repo

# Install project dependencies
!export PATH="$HOME/.local/bin:$PATH" && uv sync

In [ ]:
# Verify GPU is available
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Download labeled dataset

Add your labeled dataset as a Kaggle input dataset. Update the dataset ID below.

In [ ]:
# ============================================================
# TODO: Replace with your actual Kaggle dataset ID after upload
# ============================================================
KAGGLE_DATASET_ID = "YOUR_USERNAME/tcp-symbol-detection-v1"  # <-- CHANGE THIS

import shutil
from pathlib import Path

# Kaggle mounts input datasets at /kaggle/input/{dataset-slug}
dataset_slug = KAGGLE_DATASET_ID.split("/")[-1]
kaggle_input = Path(f"/kaggle/input/{dataset_slug}")

# Copy to writable location (Kaggle input is read-only)
data_dir = Path("/kaggle/working/repo/data/yolo")
if data_dir.exists():
    shutil.rmtree(data_dir)

shutil.copytree(kaggle_input, data_dir)

# Verify structure
images = list((data_dir / "images").rglob("*.png"))
labels = list((data_dir / "labels").rglob("*.txt"))
print(f"Images: {len(images)}")
print(f"Labels: {len(labels)}")

## 3. Tile + Train

Tiles images into 1024x1024 with 20% overlap, splits 80/20 train/val, trains YOLOv11-s for 100 epochs.

In [ ]:
import os

os.environ["PATH"] = f"{os.environ['HOME']}/.local/bin:" + os.environ["PATH"]

# Run the training pipeline
# - Tiles images from data/yolo/ -> data/yolo_tiled/
# - Splits into train/val
# - Trains YOLOv11-s for 100 epochs
#
# Adjust --batch-size if you run out of VRAM:
#   P100 (16GB): batch 16 should work
#   T4 (16GB):   batch 8-16 depending on other memory pressure
!cd /kaggle/working/repo && uv run python -m src.detection.train \
    --data-dir data/yolo \
    --tiled-dir data/yolo_tiled \
    --runs-dir runs/detect \
    --epochs 100 \
    --batch-size 16 \
    --imgsz 1024 \
    --model-size s

## 4. Review results

In [ ]:
# Find the most recent run
import glob
from pathlib import Path

run_dirs = sorted(glob.glob("runs/detect/run_*"))
if not run_dirs:
    print("No training runs found!")
else:
    run_dir = Path(run_dirs[-1])
    print(f"Latest run: {run_dir}\n")

    # Print summary
    summary = run_dir / "summary.md"
    if summary.exists():
        print(summary.read_text())

    # Display training curves
    from IPython.display import Image, display

    results_img = run_dir / "results.png"
    if results_img.exists():
        display(Image(filename=str(results_img), width=900))

    # Display confusion matrix
    cm_img = run_dir / "confusion_matrix.png"
    if cm_img.exists():
        print("\nConfusion Matrix:")
        display(Image(filename=str(cm_img), width=600))

## 5. Save best weights as Kaggle output

The best weights will be saved to `/kaggle/working/` so they appear as a Kaggle output dataset. You can then reference them from other notebooks or download them.

In [ ]:
import shutil
from pathlib import Path

# Copy best weights to Kaggle output root
output_dir = Path("/kaggle/working/tcp_model_weights")
output_dir.mkdir(exist_ok=True)

run_dirs = sorted(Path("runs/detect").glob("run_*"))
if run_dirs:
    run_dir = run_dirs[-1]
    best_pt = run_dir / "weights" / "best.pt"
    last_pt = run_dir / "weights" / "last.pt"
    summary_md = run_dir / "summary.md"

    for f in [best_pt, last_pt, summary_md]:
        if f.exists():
            shutil.copy2(f, output_dir / f.name)
            print(f"Copied: {f.name}")

    # Also copy data.yaml for reproducibility
    data_yaml = Path("data/yolo_tiled/data.yaml")
    if data_yaml.exists():
        shutil.copy2(data_yaml, output_dir / "data.yaml")

    print(f"\nOutput saved to {output_dir}")
    print("This will appear as a Kaggle dataset output after the notebook completes.")
else:
    print("No training runs found to export.")